# Tire Grip Analysis: Total G vs Tire Pressure & Temperature

This notebook analyzes the relationship between total acceleration (combined lateral and longitudinal G-forces) and tire pressure or temperature for each corner of the car.

## What You'll Find Here

- **2x2 Grip Envelope Plots**: Bucketed percentile of total G vs tire pressure/temperature for FL, FR, RL, RR
- **Statistics Table**: Mean, standard deviation for each corner

## Interpretation Guide

| Pattern | Meaning |
|---------|--------|
| Rising envelope | Higher tire metric correlates with more grip |
| Falling envelope | Higher tire metric correlates with less grip |
| Flat envelope | Tire metric has little influence on grip in this range |
| Front/rear difference | May indicate tire pressure balance issue |
| Left/right difference | May indicate uneven tire wear or pressure |

## Using Your Own Data

1. **Run the first cell** below to install packages and display the upload widget
2. **Click "Choose File"** to select your `.xrk` or `.xrz` file
3. **Configure channel names** if your TPMS channels have different names
4. **Run all remaining cells** to analyze your data

## Requirements

- Lateral and longitudinal acceleration channels (e.g., `LateralAcc`, `InlineAcc`)
- TPMS pressure or temperature channels (e.g., `TPMS_Press_LF`, `TPMS_Temp_LF`)

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [1]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q motorsports-data-notebook

# Use the Rust parser backend for ~3x faster file loading
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# Import helper functions
from motorsports_data_notebook.channels import get_top_laps
from motorsports_data_notebook.tire_grip import (
    analyze_tire_grip_multi_lap,
    format_tire_grip_stats_table,
)
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_tire_grip_scatter,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker

# Session picker with channel configuration
# Upload your own file — all laps within 103% of best are analyzed automatically
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    show_lap_picker=False,
    channel_mapping={
        "lateral_g": "LateralAcc",
        "inline_g": "InlineAcc",
        "tpms_press_fl": "TPMS_Press_LF",
        "tpms_press_fr": "TPMS_Press_RF",
        "tpms_press_rl": "TPMS_Press_LR",
        "tpms_press_rr": "TPMS_Press_RR",
        "tpms_temp_fl": "TPMS_Temp_LF",
        "tpms_temp_fr": "TPMS_Temp_RF",
        "tpms_temp_rl": "TPMS_Temp_LR",
        "tpms_temp_rr": "TPMS_Temp_RR",
    },
)
session.display()

/home/runner/work/motorsports_data_notebook/motorsports_data_notebook/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Get laps as pandas DataFrame for display
laps = session.get_laps()

In [3]:
# Display lap times table
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

,num,start_time,end_time,lap_type,lap_time
0,1,150454,279602,full,2:09.148
1,2,279602,406240,full,2:06.638
2,3,406240,532797,full,2:06.557
3,4,532797,659283,full,2:06.486
4,5,659283,787773,full,2:08.490
5,6,787773,913776,full,2:06.003
6,7,913776,1041398,full,2:07.622
7,8,1041398,1168323,full,2:06.925
8,9,1168323,1294676,full,2:06.353
9,10,1294676,1420573,full,2:05.897


## Configuration

Choose whether to analyze tire **pressure** or **temperature** by changing `metric_mode` below.

In [4]:
# Change metric_mode to "temperature" to plot tire temperature instead of pressure
metric_mode = "pressure"  # "pressure" or "temperature"

# Percentile of total G to compute per bucket (e.g. 99.9 = grip envelope)
percentile = 99.9

## Tire Grip vs Pressure/Temperature

Each subplot shows the grip envelope for one corner — the bucketed percentile (default P99.9) of total G (combined lateral + longitudinal acceleration) vs the selected tire metric. This reveals the maximum grip available at each pressure/temperature value.

In [5]:
# Run analysis across top laps (within 103% of best lap time)
log = session.get_log()
channel_names = session.get_channel_names()

top_laps = get_top_laps(laps, threshold_pct=1.03)
lap_numbers = top_laps["num"].astype(int).tolist()

print(f"Best lap time: {format_lap_time(laps['lap_time'].min())}")
print(f"Using {len(top_laps)} laps within 103% of best for analysis")

result = analyze_tire_grip_multi_lap(
    log, lap_numbers, channel_names, metric_mode=metric_mode, percentile=percentile
)

# Plot grip envelope
fig = plot_tire_grip_scatter(
    result,
    title=f"Tire Grip ({metric_mode.title()}) - Top {len(top_laps)} Laps (P{percentile})",
)
show_fig(fig)

Best lap time: 2:05.056
Using 13 laps within 103% of best for analysis


## Statistics

Per-corner summary showing mean and standard deviation for both total G and the selected tire metric.

In [6]:
# Statistics table
stats_df = format_tire_grip_stats_table(result)
# Format all numeric columns to 2 decimal places
float_cols = {col: "{:.2f}" for col in stats_df.columns if col != "Corner"}
stats_df.style.format(float_cols)  # type: ignore[arg-type]

,Corner,Mean Accel (g),Std Accel (g),Mean Pressure (bar),Std Pressure (bar)
0,FL,0.66,0.45,1.87,0.04
1,FR,0.66,0.45,1.86,0.04
2,RL,0.66,0.45,1.87,0.03
3,RR,0.66,0.45,1.86,0.03
